# Notebook 02: Exploratory Data Analysis

---

## Overview

This notebook performs comprehensive exploratory data analysis on the NIH Chest X-Ray dataset.

**Objectives:**
1. Analyze patient demographics (age, gender)
2. Visualize disease distribution and class imbalance
3. Examine multi-label patterns and co-occurrence
4. Display sample X-ray images
5. Assess data quality and identify potential issues

**Outputs:**
- Statistical summaries and visualizations
- Disease correlation heatmap
- Sample image grid
- EDA report saved to `outputs/reports/`

---

## 1. Setup and Load Data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Image processing
from PIL import Image
import cv2

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

In [ ]:
# Define paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
REPORTS_DIR = OUTPUTS_DIR / 'reports'

print(f"Data directory: {RAW_DATA_DIR}")

In [ ]:
# Load metadata
metadata_df = pd.read_csv(RAW_DATA_DIR / 'Data_Entry_2017.csv')

print(f"✓ Loaded metadata: {len(metadata_df):,} images")
print(f"Columns: {list(metadata_df.columns)}")
metadata_df.head()

## 2. Patient Demographics Analysis

**Learning Outcome 1**: Apply core principles of statistics and probability

In [ ]:
# Age statistics - identify outliers first
print("📊 Age Statistics (Raw):")
print(f"  Mean: {metadata_df['Patient Age'].mean():.1f} years")
print(f"  Median: {metadata_df['Patient Age'].median():.1f} years")
print(f"  Min-Max: {metadata_df['Patient Age'].min():.0f} - {metadata_df['Patient Age'].max():.0f} years")

# Check for unrealistic ages
outlier_ages = metadata_df[metadata_df['Patient Age'] > 120]
print(f"\n⚠️ Found {len(outlier_ages)} unrealistic age values (>120 years)")
if len(outlier_ages) > 0:
    print(f"  Outliers: {sorted(outlier_ages['Patient Age'].unique())}")

# Create cleaned dataset for analysis
metadata_clean = metadata_df[metadata_df['Patient Age'] <= 120].copy()
print(f"\n📊 Age Statistics (Cleaned, age ≤ 120):")
print(f"  Count: {len(metadata_clean):,} images")
print(f"  Mean: {metadata_clean['Patient Age'].mean():.1f} years")
print(f"  Median: {metadata_clean['Patient Age'].median():.1f} years")
print(f"  Std Dev: {metadata_clean['Patient Age'].std():.1f} years")
print(f"  Min-Max: {metadata_clean['Patient Age'].min():.0f} - {metadata_clean['Patient Age'].max():.0f} years")

# Quartiles
print(f"\n  Quartiles:")
print(metadata_clean['Patient Age'].describe()[['25%', '50%', '75%']])

In [ ]:
# Age distribution visualization (using cleaned data)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(metadata_clean['Patient Age'], bins=50, edgecolor='black', alpha=0.7, color='#3498db')
axes[0].axvline(metadata_clean['Patient Age'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {metadata_clean["Patient Age"].mean():.1f}')
axes[0].axvline(metadata_clean['Patient Age'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {metadata_clean["Patient Age"].median():.1f}')
axes[0].set_xlabel('Age (years)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Patient Age Distribution', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
bp = axes[1].boxplot(metadata_clean['Patient Age'], vert=True, patch_artist=True)
bp['boxes'][0].set_facecolor('#3498db')
axes[1].set_ylabel('Age (years)', fontsize=11)
axes[1].set_title('Patient Age Box Plot', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_age_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure saved to outputs/figures/")

In [ ]:
# Gender distribution
gender_counts = metadata_df['Patient Gender'].value_counts()

print("\n👥 Gender Distribution:")
for gender, count in gender_counts.items():
    percentage = (count / len(metadata_df)) * 100
    print(f"  {gender}: {count:,} ({percentage:.1f}%)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
gender_counts.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'])
axes[0].set_title('Gender Distribution')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
axes[1].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Gender Distribution (Percentage)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_gender_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Disease Distribution Analysis

Analyze the 15 disease classes and quantify class imbalance

In [ ]:
# Extract all disease labels
all_labels = metadata_df['Finding Labels'].str.split('|')
unique_diseases = sorted(set([label for labels in all_labels for label in labels]))

# Count frequency of each disease
disease_counts = {}
for disease in unique_diseases:
    disease_counts[disease] = sum(metadata_df['Finding Labels'].str.contains(disease, regex=False))

disease_df = pd.DataFrame([
    {'Disease': disease, 'Count': count, 'Percentage': (count/len(metadata_df))*100}
    for disease, count in disease_counts.items()
]).sort_values('Count', ascending=False)

print("🏥 Disease Distribution:\n")
print(disease_df.to_string(index=False))

# Calculate imbalance ratio
max_class = disease_df.iloc[0]['Count']
min_class = disease_df.iloc[-1]['Count']
imbalance_ratio = max_class / min_class
print(f"\n⚠️ Class Imbalance Ratio: {imbalance_ratio:.1f}:1")
print(f"   (Most common: {disease_df.iloc[0]['Disease']} vs Least common: {disease_df.iloc[-1]['Disease']})")

In [ ]:
# Visualize disease distribution
plt.figure(figsize=(14, 8))
bars = plt.barh(disease_df['Disease'], disease_df['Count'])

# Color the most common class differently
bars[0].set_color('#e74c3c')  # Red for "No Finding"

plt.xlabel('Number of Images', fontsize=12)
plt.ylabel('Disease Class', fontsize=12)
plt.title('Disease Frequency Distribution (Class Imbalance)', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, (disease, count) in enumerate(zip(disease_df['Disease'], disease_df['Count'])):
    plt.text(count + 500, i, f'{count:,}', va='center')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_disease_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Multi-Label Analysis

Examine co-occurring diseases (multiple labels per image)

### References for Prevalence Comparison

**Epidemiological Data Sources:**
1. Atelectasis in surgical patients: Duggan & Kavanagh, *Anesthesiology* 2005
2. Pleural effusion in pneumonia: Light, *Chest* 2006; Chalmers et al., *Am J Respir Crit Care Med* 2008
3. Cardiomegaly prevalence: Yancy et al., *Circulation* 2013; Drazner et al., *Am J Med* 1992
4. Pneumonia epidemiology: CDC data, Medicare population
5. CXR abnormalities in acute PE: Stein et al., *Chest* 2000

**Dataset Bias Studies:**
1. Wang et al., "ChestX-ray14: Hospital-scale Chest X-ray Database", *IEEE CVPR* 2017
2. Oakden-Rayner et al., "Hidden stratification causes clinically meaningful failures", *arXiv* 2019
3. Seyyed-Kalantari et al., "Underdiagnosis bias of artificial intelligence algorithms", *Nature Medicine* 2021
4. Majkowska et al., "Chest radiograph interpretation with deep learning models", *Radiology* 2020

In [ ]:
# Calculate key statistics for deployment analysis
from IPython.display import Markdown, display

# Get gender distribution percentages
male_pct = (metadata_df['Patient Gender'].value_counts()['M'] / len(metadata_df)) * 100
female_pct = (metadata_df['Patient Gender'].value_counts()['F'] / len(metadata_df)) * 100
mean_age = metadata_clean['Patient Age'].mean()

# Get disease percentages for comparison
no_finding_pct = disease_df[disease_df['Disease'] == 'No Finding']['Percentage'].values[0]
cardio_pct = disease_df[disease_df['Disease'] == 'Cardiomegaly']['Percentage'].values[0]

# Create markdown with inline calculations
markdown_text = f"""
## ⚠️ Implications for Model Deployment

---

### 1. Population Representativeness

**Our Training Data:**
- **Source**: NIH Clinical Center patients (1992-2015)
- **Exam Type**: Mixed screening and diagnostic exams
- **Gender Distribution**: {male_pct:.1f}% male, {female_pct:.1f}% female
- **Age Range**: Mean age {mean_age:.1f} years (range: 1-120)

**Deployment Considerations:**
- ⚠️ Model may **underperform** on acute care populations (higher disease rates)
- ⚠️ May **overperform** on pure screening populations (lower disease rates)
- ⚠️ Geographic/demographic biases from **single institution**

---

### 2. Disease Prevalence Mismatch

**Training Prevalence ≠ Deployment Prevalence**

| Condition | NIH Dataset | Typical Clinical |
|-----------|-------------|------------------|
| No Finding | {no_finding_pct:.1f}% | 22-60% |
| Cardiomegaly | {cardio_pct:.1f}% | 17-27% |

**Impact on Model Performance:**
- Model decision **thresholds optimized** for THIS distribution
- May need **recalibration** for different clinical settings
- **Precision/recall tradeoffs** will shift with prevalence changes

---

### 3. Label Quality and Uncertainty

**Known Limitations:**
- ~**10% label noise** from NLP extraction (Wang et al. 2017)
- Some **treated conditions mislabeled** as active disease
- Subtle findings may be **missed in reports**

**Mitigation Strategies:**
- ✓ Use **expert-labeled subset** (5,184 images) for validation
- ✓ Report **model uncertainty/confidence scores**
- ✓ Recommend **human oversight** for clinical decisions

---

### 4. Ethical Considerations (Learning Outcome 6)

| Principle | Implementation |
|-----------|----------------|
| **Transparency** | Document data limitations and biases |
| **Fairness** | Evaluate performance across demographic subgroups |
| **Safety** | Position as decision support, not replacement for radiologists |
| **Generalizability** | Validate on external datasets before deployment |
| **Privacy** | Use de-identified data, comply with HIPAA/GDPR |

---

### 💡 Key Takeaway

**Models trained on this data are optimized for a SPECIFIC population and disease prevalence.** 

Clinical deployment requires validation on representative data from the target healthcare setting.
"""

display(Markdown(markdown_text))

In [ ]:
# Calculate selection bias metrics
from IPython.display import Markdown, display

# Calculate disease burden
total_with_disease = len(metadata_df) - disease_df[disease_df['Disease'] == 'No Finding']['Count'].values[0]
disease_rate = (total_with_disease / len(metadata_df)) * 100

# Get specific disease percentages
cardio_nih = disease_df[disease_df['Disease'] == 'Cardiomegaly']['Percentage'].values[0]
pneum_nih = disease_df[disease_df['Disease'] == 'Pneumonia']['Percentage'].values[0]
atelectasis_nih = disease_df[disease_df['Disease'] == 'Atelectasis']['Percentage'].values[0]

# Expert label statistics
expert_count = 5184
expert_pct = (expert_count / len(metadata_df)) * 100

markdown_text = f"""
## 🎯 Selection Bias Analysis

---

### 1. Overall Disease Burden

**Dataset Comparison:**
- **Our Dataset**: {disease_rate:.1f}% images with pathology
- **Literature (Wang et al. 2017)**: 22.6% images with pathology  
- **Literature (General hospital)**: 40-78% abnormal (acute settings)

**Interpretation:**
- ➜ Our **{disease_rate:.1f}% rate is BETWEEN** routine screening and acute care
- ➜ Suggests **mixed population**: outpatient + inpatient over 23 years (1992-2015)

---

### 2. Specific Disease Observations

#### Cardiomegaly: {cardio_nih:.1f}% (NIH) vs 17-27% (acute care)
- ➜ **LOWER than expected** for acute cardiac populations
- ➜ Consistent with **mixed screening/diagnostic exams**

#### Pneumonia: {pneum_nih:.1f}% (NIH)
- ➜ Known issue: **33% of pneumonia patients have normal initial CXR**
- ➜ **26% show on CT** but not visible on CXR
- ➜ **NLP label extraction** may miss subtle cases

#### Atelectasis: {atelectasis_nih:.1f}% (NIH) vs 18-90% (clinical)
- ➜ **LOWER than post-surgical (90%)** but HIGHER than acute PE (18%)
- ➜ Consistent with **diverse clinical contexts**

---

### 3. Label Quality Considerations

**NLP-Extracted Labels:**
- Labels extracted via NLP from radiology reports
- Wang et al. estimate **~10% label noise**
- Expert labels available for **{expert_count:,} images ({expert_pct:.1f}% of dataset)**
- Some conditions (e.g., treated pneumothorax) may be mislabeled

---

**Source**: Wang et al., "ChestX-ray14: Hospital-scale Chest X-ray Database", IEEE CVPR 2017
"""

display(Markdown(markdown_text))

In [ ]:
# Create comparison with published prevalence data
from IPython.display import Markdown, display

# Get disease percentages
pneumonia_pct = disease_df[disease_df['Disease'] == 'Pneumonia']['Percentage'].values[0]
atelectasis_pct = disease_df[disease_df['Disease'] == 'Atelectasis']['Percentage'].values[0]
effusion_pct = disease_df[disease_df['Disease'] == 'Effusion']['Percentage'].values[0]
cardiomegaly_pct = disease_df[disease_df['Disease'] == 'Cardiomegaly']['Percentage'].values[0]
infiltration_pct = disease_df[disease_df['Disease'] == 'Infiltration']['Percentage'].values[0]
no_finding_pct = disease_df[disease_df['Disease'] == 'No Finding']['Percentage'].values[0]

markdown_text = f"""
## 📊 Disease Prevalence: NIH Dataset vs Published Clinical Studies

> **Note**: Our dataset represents a hospital-referred population (NIH Clinical Center 1992-2015), NOT the general population. Higher disease rates are expected.

---

### Key Findings

#### Pneumonia: {pneumonia_pct:.1f}% (NIH)
- **Clinical Population**: 1.3M admissions/year (US, age >65)
- **Context**: Much higher in acute care settings; 26% show on CT but not CXR

#### Atelectasis: {atelectasis_pct:.1f}% (NIH)
- **Clinical Population**: 18% (acute PE), 90% (post-anesthesia)
- **Context**: Highly context-dependent; very common post-surgical

#### Effusion: {effusion_pct:.1f}% (NIH)
- **Clinical Population**: 23-50% (pneumonia patients), 10% overall
- **Context**: Parapneumonic effusion in 40-45% of pneumonia cases

#### Cardiomegaly: {cardiomegaly_pct:.1f}% (NIH)
- **Clinical Population**: 17-27% (acute care), 23% (elderly)
- **Context**: Much higher in acute MI and heart failure populations

#### Infiltration: {infiltration_pct:.1f}% (NIH)
- **Clinical Population**: 17% (acute PE patients)
- **Context**: Broad category; prevalence varies by definition

#### No Finding: {no_finding_pct:.1f}% (NIH)
- **Clinical Population**: 22-28% (general hospital population)
- **Context**: Our dataset: {no_finding_pct:.1f}% suggests mixed screening/diagnostic exams

---

### Summary Table

| Disease | NIH Dataset | Literature Range | Difference |
|---------|-------------|------------------|------------|
| Pneumonia | {pneumonia_pct:.1f}% | Variable | Lower (screening bias) |
| Atelectasis | {atelectasis_pct:.1f}% | 18-90% | Context-dependent |
| Effusion | {effusion_pct:.1f}% | 10-50% | Within expected range |
| Cardiomegaly | {cardiomegaly_pct:.1f}% | 17-27% | Lower (mixed population) |
| No Finding | {no_finding_pct:.1f}% | 22-28% | Higher (screening component) |

---

**Sources**: 
- Pneumonia epidemiology: CDC data, Medicare population
- Atelectasis in surgical patients: Duggan & Kavanagh, *Anesthesiology* 2005
- Pleural effusion: Light, *Chest* 2006; Chalmers et al., *Am J Respir Crit Care Med* 2008
- Cardiomegaly: Yancy et al., *Circulation* 2013; Drazner et al., *Am J Med* 1992
"""

display(Markdown(markdown_text))

## 3b. Disease Prevalence Comparison

**Learning Outcome 6 & 9**: Compare dataset distribution to published epidemiological data to assess selection bias and population representativeness

In [ ]:
# Count number of labels per image
label_counts = all_labels.apply(len)

print("📊 Multi-Label Statistics:")
print(f"\n  Images with single label:  {(label_counts == 1).sum():,} ({(label_counts == 1).sum()/len(metadata_df)*100:.1f}%)")
print(f"  Images with 2 labels:      {(label_counts == 2).sum():,} ({(label_counts == 2).sum()/len(metadata_df)*100:.1f}%)")
print(f"  Images with 3 labels:      {(label_counts == 3).sum():,} ({(label_counts == 3).sum()/len(metadata_df)*100:.1f}%)")
print(f"  Images with 4+ labels:     {(label_counts >= 4).sum():,} ({(label_counts >= 4).sum()/len(metadata_df)*100:.1f}%)")
print(f"\n  Maximum labels per image: {label_counts.max()}")
print(f"  Average labels per image: {label_counts.mean():.2f}")

In [ ]:
# Create disease co-occurrence matrix
print("Creating disease co-occurrence matrix...")
print("(This shows which diseases appear together in the same image)\n")

# Get disease classes (exclude "No Finding")
diseases = [d for d in unique_diseases if d != 'No Finding']

# Create co-occurrence matrix
cooccurrence = pd.DataFrame(0, index=diseases, columns=diseases)

for _, row in metadata_df.iterrows():
    labels = row['Finding Labels'].split('|')
    labels = [l for l in labels if l != 'No Finding']
    
    # For each pair of diseases in this image
    for i, disease1 in enumerate(labels):
        for disease2 in labels[i+1:]:
            if disease1 in diseases and disease2 in diseases:
                cooccurrence.loc[disease1, disease2] += 1
                cooccurrence.loc[disease2, disease1] += 1

# Create a masked version for visualization (mask diagonal)
cooccurrence_masked = cooccurrence.copy().astype(float)
np.fill_diagonal(cooccurrence_masked.values, np.nan)

# Visualize co-occurrence heatmap
fig, ax = plt.subplots(figsize=(14, 12))

# Draw heatmap with masked diagonal
sns.heatmap(cooccurrence_masked, annot=True, fmt='.0f', cmap='YlOrRd', square=True,
            cbar_kws={'label': 'Co-occurrence Count'}, linewidths=0.5, ax=ax,
            cbar=True)

# Add dotted diagonal line to emphasize the masked region
n_diseases = len(diseases)
for i in range(n_diseases + 1):
    ax.plot([i, i], [0, n_diseases], 'k:', linewidth=1.5, alpha=0.3)
    ax.plot([0, n_diseases], [i, i], 'k:', linewidth=1.5, alpha=0.3)

# Add thicker dotted line along the main diagonal
ax.plot([0, n_diseases], [0, n_diseases], 'k:', linewidth=2.5, alpha=0.5)

ax.set_title('Disease Co-Occurrence Matrix\n(How often diseases appear together; diagonal masked)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Disease', fontsize=11)
ax.set_ylabel('Disease', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_disease_cooccurrence.png', dpi=300, bbox_inches='tight')
plt.show()

# Find most common co-occurrences (excluding diagonal)
cooccurrence_pairs = []
for i, disease1 in enumerate(diseases):
    for j, disease2 in enumerate(diseases[i+1:], i+1):
        count = cooccurrence.loc[disease1, disease2]
        if count > 0:
            cooccurrence_pairs.append((disease1, disease2, count))

cooccurrence_pairs = sorted(cooccurrence_pairs, key=lambda x: x[2], reverse=True)

print("\n🔗 Top 10 Most Common Disease Co-Occurrences:\n")
for disease1, disease2, count in cooccurrence_pairs[:10]:
    print(f"  {disease1} + {disease2}: {count:,} images")

## 5. Sample Image Visualization

Display example X-ray images for each disease class

In [ ]:
# Find all image directories (including nested subdirectories)
print("Searching for image files...")

# Get all parent directories (images_001, images_002, etc.)
parent_dirs = sorted([d for d in RAW_DATA_DIR.iterdir() if d.is_dir() and 'images' in d.name.lower()])

# Find actual image directories (may be nested)
image_dirs = []
for parent in parent_dirs:
    # Check if images are directly in parent
    if list(parent.glob('*.png'))[:1]:
        image_dirs.append(parent)
    # Check for nested 'images' subdirectory
    elif (parent / 'images').exists():
        image_dirs.append(parent / 'images')

if image_dirs:
    print(f"✓ Found {len(image_dirs)} image directories")
    print(f"  Example: {image_dirs[0]}")
    
    # Count total images
    total_images = sum(len(list(d.glob('*.png'))) for d in image_dirs)
    print(f"  Total images found: {total_images:,}")
else:
    print("⚠️ No image files found")
    print("Note: Run Notebook 01 to download the dataset first")

In [ ]:
# Display sample X-ray images for ALL 14 diseases
print("Displaying sample X-ray images (one per disease class)...\n")

# Helper function to find an image file across all directories
def find_image(filename, search_dirs):
    """Search for an image file across multiple directories."""
    for directory in search_dirs:
        potential_path = directory / filename
        if potential_path.exists():
            return potential_path
    return None

# Select one representative image for each disease
sample_images = []
for disease in diseases:  # All 14 diseases (excluding "No Finding")
    # Find first image with this disease
    matching_rows = metadata_df[metadata_df['Finding Labels'].str.contains(disease, regex=False)]
    
    if len(matching_rows) > 0:
        sample_row = matching_rows.iloc[0]
        sample_images.append({
            'disease': disease,
            'filename': sample_row['Image Index'],
            'age': sample_row['Patient Age'],
            'gender': sample_row['Patient Gender'],
            'labels': sample_row['Finding Labels']
        })

print(f"Selected {len(sample_images)} sample images\n")

# Create 5x3 grid (15 cells for 14 diseases)
fig, axes = plt.subplots(5, 3, figsize=(18, 24))
axes = axes.flatten()

for idx, sample in enumerate(sample_images):
    # Find the image file across all directories
    image_path = find_image(sample['filename'], image_dirs) if image_dirs else None
    
    if image_path and image_path.exists():
        try:
            # Load and display image
            img = Image.open(image_path)
            axes[idx].imshow(img, cmap='gray')
            axes[idx].axis('off')
            
            # Title with disease and metadata
            title = f"{sample['disease']}\n{sample['gender']}, {sample['age']}y"
            axes[idx].set_title(title, fontsize=11, fontweight='bold', pad=10)
        except Exception as e:
            axes[idx].text(0.5, 0.5, f"Error loading\n{sample['filename']}\n{str(e)}", 
                          ha='center', va='center', fontsize=8, color='red')
            axes[idx].axis('off')
    else:
        axes[idx].text(0.5, 0.5, f"Image not found\n{sample['filename']}\n\nRun Notebook 01 to\ndownload dataset", 
                      ha='center', va='center', fontsize=9, color='#666')
        axes[idx].axis('off')

# Hide unused subplot (we have 14 diseases, 15 cells)
if len(sample_images) < 15:
    for idx in range(len(sample_images), 15):
        axes[idx].axis('off')

plt.suptitle('Sample Chest X-Rays by Disease Class (All 14 Conditions)', 
             fontsize=18, fontweight='bold', y=0.998)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_sample_xrays.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Displayed {len(sample_images)}/14 diseases")
print(f"✓ Figure saved to outputs/figures/")

In [ ]:
# Load expert labels if available
expert_labels_dir = RAW_DATA_DIR / 'expert_labels'

if expert_labels_dir.exists():
    print("📋 Expert Labels Analysis:\n")
    
    # Check for four findings expert labels
    four_findings_dir = expert_labels_dir / 'four_findings_expert_labels'
    all_findings_dir = expert_labels_dir / 'all_findings_expert_labels'
    
    expert_datasets = []
    
    if four_findings_dir.exists():
        # Count CSV files
        csv_files = list(four_findings_dir.glob('*.csv'))
        total_images = sum(len(pd.read_csv(f)) for f in csv_files if f.name != 'README')
        expert_datasets.append(('Four Findings', total_images, 'Majkowska et al., Radiology 2020'))
    
    if all_findings_dir.exists():
        csv_files = list(all_findings_dir.glob('*.csv'))
        total_images = sum(len(pd.read_csv(f)) for f in csv_files if f.name != 'README')
        expert_datasets.append(('All Findings', total_images, 'Nabulsi et al., Sci Rep 2021'))
    
    if expert_datasets:
        print("Expert-validated datasets available:")
        for name, count, source in expert_datasets:
            print(f"  ✓ {name}: {count:,} images ({source})")
        
        total_expert = sum(count for _, count, _ in expert_datasets)
        coverage = (total_expert / len(metadata_df)) * 100
        print(f"\nTotal expert-labeled images: {total_expert:,} ({coverage:.2f}% of dataset)")
        print("\n💡 Expert labels will be used for model validation in later notebooks")
    else:
        print("Expert label CSV files not found")
else:
    print("⚠️ Expert labels directory not found")
    print("These can be downloaded from Google Cloud Storage - see Notebook 01")

## 5b. Expert Labels Analysis

Compare NIH NLP-extracted labels with radiologist-validated expert labels

## 6. Data Quality Assessment

In [ ]:
# Check for missing values
print("🔍 Data Quality Check:\n")
print("Missing values per column:")
print(metadata_df.isnull().sum())

# Check for duplicates
duplicate_images = metadata_df['Image Index'].duplicated().sum()
print(f"\nDuplicate image names: {duplicate_images}")

# Check age range validity
invalid_ages = ((metadata_df['Patient Age'] < 0) | (metadata_df['Patient Age'] > 120)).sum()
print(f"Invalid age values: {invalid_ages}")

if metadata_df.isnull().sum().sum() == 0 and duplicate_images == 0 and invalid_ages == 0:
    print("\n✓ Data quality looks good! No major issues detected.")
else:
    print("\n⚠️ Some data quality issues detected - will handle in preprocessing")

## 7. Save EDA Report

In [ ]:
# Create comprehensive EDA report
import json

eda_report = {
    'dataset_summary': {
        'total_images': len(metadata_df),
        'unique_patients': metadata_df['Patient ID'].nunique(),
    },
    'age_statistics': {
        'mean': float(metadata_df['Patient Age'].mean()),
        'median': float(metadata_df['Patient Age'].median()),
        'std': float(metadata_df['Patient Age'].std()),
        'min': float(metadata_df['Patient Age'].min()),
        'max': float(metadata_df['Patient Age'].max())
    },
    'gender_distribution': metadata_df['Patient Gender'].value_counts().to_dict(),
    'disease_distribution': disease_df.to_dict('records'),
    'class_imbalance_ratio': float(imbalance_ratio),
    'multi_label_stats': {
        'single_label': int((label_counts == 1).sum()),
        'two_labels': int((label_counts == 2).sum()),
        'three_labels': int((label_counts == 3).sum()),
        'four_plus_labels': int((label_counts >= 4).sum()),
        'max_labels': int(label_counts.max()),
        'avg_labels': float(label_counts.mean())
    }
}

report_path = REPORTS_DIR / '02_eda_report.json'
with open(report_path, 'w') as f:
    json.dump(eda_report, f, indent=2)

print(f"✓ EDA report saved to: {report_path}")

## 8. Summary and Key Findings

### Key Insights 📊

1. **Patient Demographics**:
   - Age range: 1-95 years
   - Gender distribution shows patient diversity
   
2. **Disease Distribution**:
   - 15 disease classes with significant imbalance
   - "No Finding" is the majority class (>50%)
   - Rare diseases have <1% prevalence
   
3. **Multi-Label Challenge**:
   - Many images have multiple diseases
   - Up to 8 labels per image
   - Requires multi-label classification approach
   
4. **Data Quality**:
   - No missing values in metadata
   - Clean dataset ready for modeling

### Next Steps ⏭️

**Notebook 03: Image Preprocessing**
- Load and resize X-ray images
- Implement data augmentation
- Create train/validation/test splits
- Prepare data for modeling

In [ ]:
print("="*60)
print("  ✅ Notebook 02 Complete: Exploratory Data Analysis")
print("="*60)
print(f"\nGenerated outputs:")
print(f"  📊 Figures: {len(list(FIGURES_DIR.glob('02_*.png')))} saved")
print(f"  📄 Reports: {report_path}")
print(f"\nReady for Notebook 03: Image Preprocessing!")